## Search Engine with Tools and agents

In [1]:
## arxiv searhc
## tools creation
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper , WikipediaAPIWrapper

In [2]:
# use the inbuilt tool of wikipedia
wiki_api_wrapper = WikipediaAPIWrapper(top_k_results=1 , doc_content_chars_max= 250)
wiki = WikipediaQueryRun(api_wrapper= wiki_api_wrapper)
wiki.name

'wikipedia'

In [3]:
arxiv_api_wrapper = ArxivAPIWrapper(top_k_results= 1,doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper= arxiv_api_wrapper)
arxiv.name

'arxiv'

In [4]:
tools = [wiki, arxiv]

In [5]:
# custom tool [RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

/Users/rachin/Desktop/GenAI/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
loader = WebBaseLoader("https://www.geeksforgeeks.org/deep-learning/deep-learning-tutorial/")
docs = loader.load()
text_splitter= RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents = text_splitter.split_documents(docs)
embedding = HuggingFaceEmbeddings(model_name="ibm-granite/granite-embedding-97m-multilingual-r2")
vectordb = FAISS.from_documents(documents,embedding)
retriever = vectordb.as_retriever()

Loading weights: 100%|██████████| 74/74 [00:00<00:00, 15989.83it/s]


In [20]:
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(retriever,"deep_learning", "Search and retrieve information about deep learning. Input must be a single search query string.")

retriever_tool.name

'deep_learning'

In [8]:
tools = [wiki, arxiv, retriever_tool]

In [25]:
## Run all this tools with Agents and LLM Models

from langchain_groq import ChatGroq
from  dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(model = "llama-3.3-70b-versatile")

In [27]:
from langchain.agents import create_agent


agent = create_agent(
    model=llm, 
    tools=tools, 
    system_prompt="You are a helpful assistant"
)


In [28]:
# Pass your query as a list of conversation messages
response = agent.invoke({
    "messages": [{"role": "user", "content": "tell me about deep learning"}]
})

# Print out the final answer from the assistant
print(response["messages"][-1].content)


Deep learning is a subset of artificial intelligence (AI) and machine learning (ML) that uses neural networks to analyze and interpret data. It's commonly used for tasks like image recognition, speech processing, and natural language understanding. Some key concepts in deep learning include neural networks, convolutional neural networks (CNNs), recurrent neural networks (RNNs), and generative models.


In [18]:
response = agent.invoke({
    "messages":[{"role":"user","content":"tell me about paper 1706.03762 ?"}]
})
print(response["messages"][-1].content)

The attention mechanism is a type of neural network component that allows the model to focus on different parts of the input data when generating the output.


In [30]:
response = agent.invoke({
    "messages":[{"role":"user","content":"What is transformer ?"}]
})
print(response["messages"][-1].content)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ks7n2mawf5xtrbcz2s2p69gx` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99245, Requested 10864. Please try again in 2h25m34.175999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Failed to refresh cache entry hwchase17/openai-functions-agent: Connection error caused failure to GET /commits/hwchase17/openai-functions-agent/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/openai-functions-agent/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: None
API Key: lsv2_********************************************35
Failed to refresh cache entry hwchase17/openai-functions-agent: Connection error caused failure to GET /commits/hwchase17/openai-functions-agent/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded w